In [ ]:
!pip install evaluate transformers accelerate
!pip install datasets==3.6.0
!pip install --upgrade wandb

In [ ]:
!wandb login --relogin "insert_your_wandb_token_here"

In [ ]:
!pip install evaluate transformers accelerate
!pip install datasets==3.6.0
!pip install --upgrade wandb

# %%
!pip install evaluate datasets transformers accelerate wandb scikit-learn

# %%
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Function
import wandb
from datetime import datetime
from datasets import load_dataset, Audio

from transformers import (
    AutoModel,
    AutoFeatureExtractor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput
from huggingface_hub import login
import evaluate
from sklearn.metrics import confusion_matrix, classification_report

# %%
current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Current time: {current_time_str}")

# check if there GPU
print("Checking if GPU is available:")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"torch.cuda.get_device_name(): {torch.cuda.get_device_name(0)}")

# %%
login(token="insert_your_huggingface_token_here")

# %%
wandb.init(
    project="Indic-SLID",
    name=f"SLID_DANN_mHuBERT_{current_time_str}"
)

# %%
model_id = "utter-project/mHuBERT-147"

feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id,
    do_normalize=True,
    return_attention_mask=True,
)

# %%
dataset = load_dataset("badrex/nnti-dataset-full")

# shuffle the dataset
train_ds = dataset['train'].shuffle(seed=42)
valid_ds = dataset['validation'].shuffle(seed=42)

# resample to 16kHz
train_ds = train_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))

# %%
# based on the model typel, set input features key
if model_id == "facebook/w2v-bert-2.0":
    input_features_key = "input_features"
else:
    input_features_key = "input_values"

max_duration = 7 # in seconds

# %%
# get the set of languages
labels_list = train_ds.unique('language')
sorted_labels = sorted(l.upper() for l in labels_list)
print(f"Languages: {sorted_labels}")

str_to_int = {s: i for i, s in enumerate(labels_list)}
int_to_str = {i: s for s, i in str_to_int.items()}
num_labels = len(str_to_int)

train_speakers = train_ds.unique('speaker_id')
valid_speakers = valid_ds.unique('speaker_id')

all_speakers_list = list(set(train_speakers + valid_speakers))

speaker_to_int = {s: i for i, s in enumerate(all_speakers_list)}
num_speakers = len(speaker_to_int)

print(f"Total unique speakers across all data: {num_speakers}")

# %%
def preprocess_function(examples):
    max_samples = int(feature_extractor.sampling_rate * max_duration)

    audio_arrays = []
    for x in examples["audio_filepath"]:
        audio_array = x["array"]
        current_length = len(audio_array)

        # random cropping
        if current_length > max_samples:
            start_idx = np.random.randint(0, current_length - max_samples)
            cropped_array = audio_array[start_idx : start_idx + max_samples]
            audio_arrays.append(cropped_array)
        else:
            audio_arrays.append(audio_array)

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        truncation=True,
        max_length=max_samples,
        return_attention_mask=True,
    )

    inputs["label"] = [str_to_int[x] for x in examples["language"]]
    inputs["speaker_labels"] = [speaker_to_int[x] for x in examples["speaker_id"]]

    inputs[input_features_key] = [np.array(x) for x in inputs[input_features_key]]
    inputs["length"] = [len(f) for f in inputs[input_features_key]]

    return inputs

keep_cols = ['speaker_id', 'language']

# %%
# ## encode the train and valid splits
train_ds_encoded = train_ds.map(
    preprocess_function,
    remove_columns=[c for c in train_ds.column_names if c not in keep_cols],
    batched=True,
    batch_size=32,
)

valid_ds_encoded = valid_ds.map(
    preprocess_function,
    remove_columns=[c for c in valid_ds.column_names if c not in keep_cols],
    batched=True,
    batch_size=32,
)

# %%
# GRL layer to multiply the gradients by a negative constant alpha during the backward pass
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)

# %%
# DANN implementation
class DANN_SLID_Model(nn.Module):
    def __init__(self, model_id, num_languages, num_speakers, alpha=0.0):
        super().__init__()

        self.base_model = AutoModel.from_pretrained(model_id)

        for param in self.base_model.feature_extractor.parameters():
            param.requires_grad = False

        for param in self.base_model.encoder.layers[:6].parameters():
            param.requires_grad = False

        hidden_size = self.base_model.config.hidden_size

        # Language head
        self.language_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_languages)
        )

        # Speaker head
        self.speaker_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(0.4),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Dropout(0.4),
            nn.Linear(hidden_size // 2, num_speakers)
        )
        self.alpha = alpha

    def forward(self, input_values, attention_mask=None, labels=None, speaker_labels=None, **kwargs):
        outputs = self.base_model(input_values=input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        if attention_mask is None:
            attention_mask = torch.ones(hidden_states.shape[:2], device=hidden_states.device)
        else:
            attention_mask = self.base_model._get_feature_vector_attention_mask(
                hidden_states.shape[1], attention_mask
            )

        # mean pooling
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        lang_logits = self.language_head(pooled_output)

        reversed_features = grad_reverse(pooled_output, self.alpha)
        speaker_logits = self.speaker_head(reversed_features)

        loss = None
        if labels is not None and speaker_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            lang_loss = loss_fct(lang_logits, labels)
            speaker_loss = loss_fct(speaker_logits, speaker_labels)

            loss = lang_loss + speaker_loss

        return SequenceClassifierOutput(loss=loss, logits=lang_logits)

slid_model = DANN_SLID_Model(model_id, num_labels, num_speakers, alpha=0.0)

# %%
# create collator for padding
class DANNDataCollator:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features):
        batch = {
            input_features_key: [f[input_features_key] for f in features],
            "attention_mask": [f["attention_mask"] for f in features]
        }

        batch = self.feature_extractor.pad(
            batch,
            padding=True,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor([f["label"] for f in features], dtype=torch.long)
        batch["speaker_labels"] = torch.tensor([f["speaker_labels"] for f in features], dtype=torch.long)

        return batch

data_collator = DANNDataCollator(feature_extractor)

# %%
batch_size = 16
gradient_accumulation_steps = 4
num_train_epochs = 15
lr = 5e-5

# %%
training_args = TrainingArguments(
    output_dir="./results_dann",
    dataloader_num_workers=4,
    report_to="wandb",
    logging_steps=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=lr,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    weight_decay=0.05,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
    remove_unused_columns=False,
)

# %%
# load evaluation metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)

    if isinstance(eval_pred.label_ids, tuple):
        true_labels = eval_pred.label_ids[0]
    else:
        true_labels = eval_pred.label_ids

    return accuracy_metric.compute(
        predictions=predictions,
        references=true_labels
    )

# %%
class DynamicAlphaCallback(TrainerCallback):
    def __init__(self, max_alpha=1.0):
        self.max_alpha = max_alpha

    def on_step_begin(self, args, state, control, model, **kwargs):
        if state.max_steps > 0:
            p = state.global_step / state.max_steps
        else:
            p = 0.0

        current_alpha = (2.0 / (1.0 + np.exp(-5.0 * p)) - 1.0) * self.max_alpha

        if hasattr(model, "module"):
            model.module.alpha = current_alpha
        else:
            model.alpha = current_alpha

        if wandb.run is not None:
            wandb.log({"train/dynamic_alpha": current_alpha}, step=state.global_step)

# %%
trainer = Trainer(
    slid_model,
    training_args,
    train_dataset=train_ds_encoded,
    eval_dataset=valid_ds_encoded,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=4),
        DynamicAlphaCallback(max_alpha=0.25)
    ]
)

# %%
print("Train loop starting...")
trainer.train()

# %%
# save model to disk
save_dir = "./indic-SLID/dann_inprogress"
os.makedirs(save_dir, exist_ok=True)

print("Final evaluation starting...", flush=True)
metrics = trainer.evaluate()
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)

# %%
print("\n" + "="*50, flush=True)
print("Generating evaluation metrics for logs", flush=True)
print("="*50, flush=True)

val_outputs = trainer.predict(valid_ds_encoded)
y_pred = np.argmax(val_outputs.predictions, axis=1)

if isinstance(val_outputs.label_ids, tuple):
    y_true = val_outputs.label_ids[0]
else:
    y_true = val_outputs.label_ids

ordered_label_names = [int_to_str[i] for i in range(num_labels)]

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=ordered_label_names, columns=ordered_label_names)

print("\nConfusion matrix", flush=True)
print(cm_df.to_string(), flush=True)

print("\nClassification report", flush=True)
report = classification_report(y_true, y_pred, target_names=ordered_label_names, zero_division=0)
print(report, flush=True)

# %%
print("\n" + "="*50, flush=True)
print("Analyzing accuracy per speaker (dann check)", flush=True)
print("="*50, flush=True)

speaker_ids = [speaker_to_int[s] for s in valid_ds_encoded["speaker_id"]]
true_languages = [int_to_str[label] for label in y_true]

results_df = pd.DataFrame({
    "Language": true_languages,
    "Speaker_ID": speaker_ids,
    "Correct": (y_true == y_pred)
})

speaker_stats = results_df.groupby(["Language", "Speaker_ID"]).agg(
    Total_Utterances=("Correct", "count"),
    Accuracy=("Correct", "mean")
).reset_index()

speaker_stats["Accuracy"] = (speaker_stats["Accuracy"] * 100).round(2).astype(str) + "%"

print("\nAccuracy by speaker", flush=True)
print(speaker_stats.to_string(index=False), flush=True)

bias_summary = results_df.groupby(["Language", "Speaker_ID"])["Correct"].mean().groupby("Language").agg(
    Min_Accuracy="min",
    Max_Accuracy="max"
).reset_index()

bias_summary["Min_Accuracy"] = (bias_summary["Min_Accuracy"] * 100).round(2).astype(str) + "%"
bias_summary["Max_Accuracy"] = (bias_summary["Max_Accuracy"] * 100).round(2).astype(str) + "%"

print("\nBias summary", flush=True)
print(bias_summary.to_string(index=False), flush=True)

# %%
print("="*50 + "\n", flush=True)

torch.save(slid_model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))

print("generating tsne plot")

# %%
# tsne visualization
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

slid_model.eval()
device = next(slid_model.parameters()).device

all_embeddings = []
all_labels = []

tsne_batch_size = 16
valid_loader = torch.utils.data.DataLoader(
    valid_ds_encoded,
    batch_size=tsne_batch_size,
    collate_fn=data_collator,
    shuffle=False,
)

with torch.no_grad():
    for batch in valid_loader:
        input_values = batch[input_features_key].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].cpu().numpy()

        # get last hidden state
        outputs = slid_model.base_model(input_values=input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        # mean pooling
        feat_mask = slid_model.base_model._get_feature_vector_attention_mask(
            hidden_states.shape[1], attention_mask
        )
        mask_expanded = feat_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        all_embeddings.append(pooled.cpu().numpy())
        all_labels.append(labels_batch)

all_embeddings = np.concatenate(all_embeddings, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

print(f"embeddings shape: {all_embeddings.shape}")

# run tsne
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

# plot colored by speaker
speaker_id_list = valid_ds_encoded["speaker_id"]
unique_speakers = sorted(set(speaker_id_list))
speaker_to_idx = {s: i for i, s in enumerate(unique_speakers)}
speaker_indices = np.array([speaker_to_idx[s] for s in speaker_id_list])

plt.figure(figsize=(14, 10))
plt.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=speaker_indices, cmap='hsv',
    alpha=0.5, s=15,
)
plt.title("t-SNE - Colored by Speaker (DANN)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tsne_by_speaker_dann.png"), dpi=150)
plt.show()

print("completed")